# Notebook #9: Spatial Immunosenescence

This notebook projects immune-cell senescence and dysfunction scores from the single-cell reference onto two peritoneal endometriosis lesions.

**Main analysis questions:**

1.   Are immunosenescenct immune states spatially correlated?
2.   Do dysfunctional cells and senescent cells cluster together?
3.   Does high immune infiltration correlate with senescence/dysfunction states?
4.   How do tissues 346 and 355 compare?



Note -- because these tissues are peritoneal lesions, I've subsetted only these tissue samples from the sc object to match the microenvironment.

_Future projects can compare signatures against other tissue types when they become available._

In [4]:
!pip install -q scanpy\
pandas\
numpy==2.0.2\
squidpy\
matplotlib\
anndata\
seaborn\
ptitprince\
igraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 12.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.5/280.5 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 53.9 MB/

In [6]:
# -- Imports
from pathlib import Path

import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import igraph
import seaborn as sns
import matplotlib.pyplot as plt
import ptitprince as pt
from matplotlib.patches import Patch


ad.settings.allow_write_nullable_strings = True
import anndata as ad

In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [8]:
# -- Paths
project_path = Path("/content/drive/MyDrive/endo-immune-atlas")

input_path_reference = (
    project_path
    / "data"
    / "interim"
    / "spatial"
    / "immune_reference_posterior.h5ad"
)

input_path_346 = (
    project_path
    / "data"
    / "interim"
    / "spatial"
    /"BEME_346_spatial_immune_architecture.h5ad"
)

input_path_355G = (
    project_path
    / "data"
    / "interim"
    / "spatial"
    /"BEME_355G_spatial_immune_architecture.h5ad"
)

output_path_data = (
    project_path
    / "data"
    / "interim"
    / "spatial")

output_path_figures = (
    project_path
    / "figures"
    / "spatial"
    / "immunosenescence")

output_path_results = (
    project_path
    / "results"
    / "spatial"
    / "immunosenescence")

output_path_data.mkdir(parents=True, exist_ok=True)
output_path_figures.mkdir(parents=True, exist_ok=True)
output_path_results.mkdir(parents=True, exist_ok=True)

In [9]:
# -- COLOR PALETTES

immune_palette_map = {
    "Immune-high": "#8B1E3F",
    "Immune-moderate-high": "#D55E00",
    "Immune-moderate": "#E69F00",
    "Immune-moderate-low": "#4C78A8",
    "Immune-low": "#5F6368",
    "Immune-poor": "#D9D9D9"
}

immune_class_order = [
    "Immune-high",
    "Immune-moderate-high",
    "Immune-moderate",
    "Immune-moderate-low",
    "Immune-low",
    "Immune-poor"
]

In [10]:
# -- Import objects
immune_posterior = sc.read_h5ad(input_path_reference)
adata_346 = sc.read_h5ad(input_path_346)
adata_355G = sc.read_h5ad(input_path_355G)

print(immune_posterior)
print(adata_346)
print(adata_355G)

AnnData object with n_obs × n_vars = 27882 × 22978
    obs: 'sample_id', 'patient_id', 'tissue_type', 'condition', 'lesion_site', 'dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribos', 'pct_counts_ribos', 'total_counts_hemos', 'pct_counts_hemos', 'n_genes', 'n_counts', 'outlier_mt', 'doublet_score', 'predicted_doublet', 'leiden_res_0.01', 'leiden_res_0.02', 'leiden_res_0.05', 'cluster_label', 'leiden_res_0.20', 'leiden_res_0.50', 'leiden_res_1.00', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'immune_cell_type', 'cell_type_short', 'lineage', 'core_sen_score', 'sasp_score', 'innate_sen_score', 'adaptive_sen_score', 'composite_sen_score', 'dysfunction_score', 'tcell_exhaustion_score', 'nk_dysfunction_score', 'macrophage_dysfunction_score', 'bcell_exhaustion_score', 'cdc1_dysfunction_score', 'cdc2_dysfunction_score', 'pdc_dysfunction_score', 'senescent_state', 'dysfunction_state', 'sen_dysfunction_label', 'cell2

In [11]:
#============================================================================
# TISSUE DICT - update this when more tissues for analysis
#============================================================================
adatas = {
    "BEME346": adata_346,
    "BEME355G": adata_355G,
}

In [12]:
# -- CALCULCATE MEAN SCORES FOR SENESCENCE AND DYSFUNCTION PER TISSUE TYPE
required_reference_columns = {
    "lesion_site",
    "composite_sen_score",
    "dysfunction_score"
}

missing_reference_columns = (
    required_reference_columns
    - set(immune_posterior.obs.columns)
)

if missing_reference_columns:
    raise KeyError(
        f"Missing reference columns: {sorted(missing_reference_columns)}"
    )

if "cell2location_label" in immune_posterior.obs.columns:
    reference_label_col = "cell2location_label"
elif "cell_type_short" in immune_posterior.obs.columns:
    reference_label_col = "cell_type_short"
else:
    raise KeyError(
        "The reference object needs either "
        "'cell2location_label' or 'cell_type_short'."
    )

peritoneal_reference = immune_posterior[
    immune_posterior.obs["lesion_site"] == "peritoneal"
].copy()

mean_peri_sen_scores = (
    peritoneal_reference.obs
    .groupby(reference_label_col, observed=True)["composite_sen_score"]
    .mean()
)

mean_peri_dys_scores = (
    peritoneal_reference.obs
    .groupby(reference_label_col, observed=True)["dysfunction_score"]
    .mean()
)

display(pd.DataFrame({
    "mean_senescence": mean_peri_sen_scores,
    "mean_dysfunction": mean_peri_dys_scores
}))

,mean_senescence,mean_dysfunction
cell2location_label,,
B,-0.158144,0.002269
CD4 T,-0.113584,-0.171170
CD8 T,-0.162290,-0.138454
DC,0.007623,-0.117662
Mono-C,0.045992,-0.161117
Mono-NC,0.036000,-0.192172
NK-CD16+,0.028296,0.017009
NK-CD16-,0.038908,0.345633
TRM,0.209060,0.220454


In [21]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTIONS
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def map_scores(
    adata,
    state_scores,
    state_col_name
):
    """Estimate an abundance-weighted immune-state score for each spot."""

    available_cell_types = [
        cell_type
        for cell_type in state_scores.index
        if cell_type in adata.obs.columns
    ]

    missing_cell_types = [
        cell_type
        for cell_type in state_scores.index
        if cell_type not in adata.obs.columns
    ]

    if not available_cell_types:
        raise ValueError(
            "No reference cell-type labels match the spatial abundance columns."
        )

    if missing_cell_types:
        print("Cell types not found:", missing_cell_types)

    scores = state_scores.loc[available_cell_types]

    weighted_score = (
        adata.obs[available_cell_types]
        .mul(scores, axis=1)
        .sum(axis=1)
    )

    total_used_abundance = (
        adata.obs[available_cell_types]
        .sum(axis=1)
        .replace(0, np.nan)
    )

    adata.obs[state_col_name] = (
        weighted_score
        .div(total_used_abundance)
    )

    return adata


def map_cluster_means(
    adata,
    cluster_col,
    score_col,
    mean_col_name
):
    """Calculate a region mean and map it back to each spot."""

    cluster_means = (
        adata.obs
        .groupby(cluster_col, observed=True)[score_col]
        .mean()
    )

    adata.obs[mean_col_name] = (
        adata.obs[cluster_col]
        .astype(str)
        .map(cluster_means)
    )

    return cluster_means


def make_cluster_summary(
    adata,
    score_col,
    mean_name,
    cluster_col="leiden_0.4"
):
    """Summarize projected score and immune class by spatial region."""

    summary = (
        adata.obs
        .groupby(cluster_col, observed=True)
        .agg(
            **{
                mean_name: (score_col, "mean"),
                "immune_class": (
                    "tissue_level_immune_class",
                    "first"
                )
            }
        )
        .reset_index()
    )

    summary["immune_class"] = summary["immune_class"].astype(str)

    summary["color"] = [
        immune_palette_map[label]
        for label in summary["immune_class"]
    ]

    return summary


def make_plot_df(
    adata,
    score_col,
    tissue_name
):
    """Create the spot-level dataframe used for distribution plots."""

    plot_df = adata.obs[
        [
            score_col,
            "tissue_level_immune_class",
            "leiden_0.4"
        ]
    ].copy()

    plot_df["tissue"] = tissue_name

    plot_df["tissue_level_immune_class"] = pd.Categorical(
        plot_df["tissue_level_immune_class"],
        categories=immune_class_order,
        ordered=True
    )

    return plot_df


def plot_spatial_scores(
    adatas,
    score_col,
    score_label,
    file_name
):
    """Plot one projected score across both lesions with a shared scale."""

    values = pd.concat([
        adata.obs[score_col]
        for adata in adatas.values()
    ])

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    for index, (tissue, adata) in enumerate(adatas.items()):

        sq.pl.spatial_scatter(
            adata,
            color=score_col,
            cmap="magma",
            size=1.5,
            vmin=values.min(),
            vmax=values.max(),
            colorbar=index == 1,
            ax=axes[index],
            title=tissue,
            img=False
        )

        axes[index].text(
            0,
            1.03,
            chr(65 + index),
            transform=axes[index].transAxes,
            fontsize=18,
            fontweight="bold"
        )

    fig.suptitle(score_label, fontsize=16)
    plt.tight_layout()

    plt.savefig(
        output_path_figures / file_name,
        bbox_inches="tight",
        dpi=300
    )

    plt.close()


def plot_abundance_relationship(
    adatas,
    score_col,
    y_label,
    file_name
):
    """Compare immune abundance with a projected immune-state score."""

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5),
        sharey=True,
        constrained_layout=True
    )

    for index, (tissue, adata) in enumerate(adatas.items()):

        sns.scatterplot(
            data=adata.obs,
            x="total_immune_spot_level",
            y=score_col,
            hue="tissue_level_immune_class",
            hue_order=immune_class_order,
            palette=immune_palette_map,
            alpha=0.7,
            legend=index == 1,
            ax=axes[index]
        )

        sns.regplot(
            data=adata.obs,
            x="total_immune_spot_level",
            y=score_col,
            scatter=False,
            lowess=True,
            color="black",
            line_kws={"lw": 3},
            ax=axes[index]
        )

        axes[index].set_title(tissue, fontsize=15)
        axes[index].set_xlabel("Total immune abundance")

        axes[index].text(
            0,
            1.03,
            chr(65 + index),
            transform=axes[index].transAxes,
            fontsize=18,
            fontweight="bold"
        )

    axes[0].set_ylabel(y_label)
    axes[1].set_ylabel("")

    if axes[1].get_legend() is not None:
        axes[1].legend(
            title=None,
            bbox_to_anchor=(1.02, 1),
            loc="upper left"
        )

    plt.savefig(
        output_path_figures / file_name,
        bbox_inches="tight",
        dpi=300
    )

    plt.close()


def plot_cluster_scores(
    summaries,
    mean_col,
    y_label,
    file_name
):
    """Plot region-level projected scores colored by immune class."""

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5),
        sharey=True
    )

    for index, (tissue, summary) in enumerate(summaries.items()):

        axes[index].bar(
            summary["leiden_0.4"].astype(str),
            summary[mean_col],
            color=summary["color"]
        )

        axes[index].set_title(tissue)
        axes[index].set_xlabel("Spatial region")

        axes[index].text(
            0,
            1.03,
            chr(65 + index),
            transform=axes[index].transAxes,
            fontsize=18,
            fontweight="bold"
        )

    axes[0].set_ylabel(y_label)
    axes[1].set_ylabel("")

    observed_classes = [
        label
        for label in immune_class_order
        if any(
            label in summary["immune_class"].values
            for summary in summaries.values()
        )
    ]

    legend_handles = [
        Patch(
            color=immune_palette_map[label],
            label=label
        )
        for label in observed_classes
    ]

    axes[1].legend(
        handles=legend_handles,
        frameon=True,
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.tight_layout()

    plt.savefig(
        output_path_figures / file_name,
        bbox_inches="tight",
        dpi=300
    )

    plt.close()


def plot_score_rainclouds(
    plot_dfs,
    score_col,
    x_label,
    file_name,
    xlim=None
):
    """Plot projected-score distributions across immune classes."""

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(15, 9),
        sharey=True
    )

    for index, (tissue, plot_df) in enumerate(
        plot_dfs.items()
    ):

        plot_df = plot_df[
            plot_df["tissue_level_immune_class"].notna()
            & plot_df[score_col].notna()
        ].copy()

        observed_classes = [
            label
            for label in immune_class_order
            if label in (
                plot_df["tissue_level_immune_class"]
                .astype(str)
                .unique()
            )
        ]

        plot_df["tissue_level_immune_class"] = pd.Categorical(
            plot_df["tissue_level_immune_class"].astype(str),
            categories=observed_classes,
            ordered=True
        )

        tissue_palette = [
            immune_palette_map[label]
            for label in observed_classes
        ]

        pt.RainCloud(
            data=plot_df,

            # PtitPrince expects the category in x
            # and the numeric measurement in y,
            # even when orient="h".
            x="tissue_level_immune_class",
            y=score_col,

            order=observed_classes,
            palette=tissue_palette,
            orient="h",

            width_viol=0.55,
            width_box=0.18,
            point_size=1.2,
            move=0.2,
            alpha=0.65,

            ax=axes[index]
        )

        axes[index].set_title(
            tissue,
            fontsize=16
        )

        axes[index].set_xlabel(
            x_label
        )

        axes[index].text(
            0,
            1.03,
            chr(65 + index),
            transform=axes[index].transAxes,
            fontsize=18,
            fontweight="bold"
        )

        if xlim is not None:
            axes[index].set_xlim(xlim)

    axes[0].set_ylabel(
        "Immune class"
    )

    axes[1].set_ylabel("")

    plt.subplots_adjust(
        wspace=0.12
    )

    plt.savefig(
        output_path_figures / file_name,
        bbox_inches="tight",
        dpi=300
    )

    plt.close()

# **SENESCENCE SECTION**

In [14]:
# -- MAP SENESCENT SCORES TO SPATIAL ARCHITECTURE
for tissue, adata in adatas.items():

    map_scores(
        adata,
        state_scores=mean_peri_sen_scores,
        state_col_name="projected_senescence_score"
    )

    print(f"Finished senescence mapping for: {tissue}")

Finished senescence mapping for: BEME346
Finished senescence mapping for: BEME355G


In [15]:
#==============================================================
# FIGURE 9_1: SENESCENCE SCORES ON SPATIAL MAP
#==============================================================

plot_spatial_scores(
    adatas,
    score_col="projected_senescence_score",
    score_label="Projected immune senescence",
    file_name="09_spatial_senescence_scores.png"
)

In [16]:
#=============================================================================
# FIGURE 9_2: CORRELATION BETWEEN PROJECTED SEN SCORES AND IMMUNE CLASS
#=============================================================================

plot_abundance_relationship(
    adatas,
    score_col="projected_senescence_score",
    y_label="Projected senescence score",
    file_name="09_immune_abundance_senescence.png"
)

In [17]:
# -- PROJECTED SENESCENCE BY SPATIAL REGION

senescence_cluster_means = {}
senescence_summaries = {}

for tissue, adata in adatas.items():

    senescence_cluster_means[tissue] = map_cluster_means(
        adata,
        cluster_col="leiden_0.4",
        score_col="projected_senescence_score",
        mean_col_name="cluster_mean_senescence"
    )

    senescence_summaries[tissue] = make_cluster_summary(
        adata,
        score_col="projected_senescence_score",
        mean_name="mean_senescence"
    )

plot_cluster_scores(
    senescence_summaries,
    mean_col="mean_senescence",
    y_label="Mean projected senescence",
    file_name="09_spatial_regions_senescence.png"
)

In [22]:
#==============================================================
# FIGURE 9_3: SENESCENT SCORES BY CLUSTER REGIONS
#==============================================================

senescence_plot_dfs = {
    tissue: make_plot_df(
        adata,
        score_col="projected_senescence_score",
        tissue_name=tissue
    )
    for tissue, adata in adatas.items()
}

plot_score_rainclouds(
    senescence_plot_dfs,
    score_col="projected_senescence_score",
    x_label="Projected senescence score",
    file_name="09_immune_class_senescence_raincloud.png"
)

/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:154: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group_name, group_df in self.plot_data.groupby(grouping_vars):
/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:1070: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:154: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group_name, group_df in self.plot_data.group

# **DYSFUNCTION SECTION**

This section repeats the same spatial comparisons using the immune dysfunction score.

In [23]:
# -- MAP DYSFUNCTION SCORES TO SPATIAL ARCHITECTURE
for tissue, adata in adatas.items():

    map_scores(
        adata,
        state_scores=mean_peri_dys_scores,
        state_col_name="projected_dysfunction_score"
    )

    print(f"Finished dysfunction mapping for: {tissue}")

Finished dysfunction mapping for: BEME346
Finished dysfunction mapping for: BEME355G


In [24]:
#==============================================================
# FIGURE 9_5: DYSFUNCTION SCORES ON SPATIAL MAP
#==============================================================

plot_spatial_scores(
    adatas,
    score_col="projected_dysfunction_score",
    score_label="Projected immune dysfunction",
    file_name="09_spatial_dysfunction_scores.png"
)

In [25]:
#=============================================================================
# FIGURE 9_6: CORRELATION BETWEEN PROJECTED SEN SCORES AND IMMUNE CLASS
#=============================================================================

plot_abundance_relationship(
    adatas,
    score_col="projected_dysfunction_score",
    y_label="Projected dysfunction score",
    file_name="09_immune_abundance_dysfunction.png"
)

In [26]:
# -- PROJECTED DYSFUNCTION BY SPATIAL REGION

dysfunction_cluster_means = {}
dysfunction_summaries = {}

for tissue, adata in adatas.items():

    dysfunction_cluster_means[tissue] = map_cluster_means(
        adata,
        cluster_col="leiden_0.4",
        score_col="projected_dysfunction_score",
        mean_col_name="cluster_mean_dysfunction"
    )

    dysfunction_summaries[tissue] = make_cluster_summary(
        adata,
        score_col="projected_dysfunction_score",
        mean_name="mean_dysfunction"
    )

plot_cluster_scores(
    dysfunction_summaries,
    mean_col="mean_dysfunction",
    y_label="Mean projected dysfunction",
    file_name="09_spatial_regions_dysfunction.png"
)

In [27]:
#==============================================================
# FIGURE 9_7: SENESCENT SCORES BY IMMUNE CLASSES
#==============================================================

dysfunction_plot_dfs = {
    tissue: make_plot_df(
        adata,
        score_col="projected_dysfunction_score",
        tissue_name=tissue
    )
    for tissue, adata in adatas.items()
}

plot_score_rainclouds(
    dysfunction_plot_dfs,
    score_col="projected_dysfunction_score",
    x_label="Projected dysfunction score",
    file_name="09_immune_class_dysfunction_raincloud.png"
)

/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:154: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group_name, group_df in self.plot_data.groupby(grouping_vars):
/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:1070: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/usr/local/lib/python3.12/dist-packages/ptitprince/PtitPrince.py:154: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group_name, group_df in self.plot_data.group

In [28]:
# -- SAVE DATA

senescence_summary_table = pd.concat(
    [
        summary.assign(library_id=tissue)
        for tissue, summary in senescence_summaries.items()
    ],
    ignore_index=True
)

dysfunction_summary_table = pd.concat(
    [
        summary.assign(library_id=tissue)
        for tissue, summary in dysfunction_summaries.items()
    ],
    ignore_index=True
)

senescence_summary_table.to_csv(
    output_path_results / "09_region_senescence_summary.csv",
    index=False
)

dysfunction_summary_table.to_csv(
    output_path_results / "09_region_dysfunction_summary.csv",
    index=False
)

output_file_346 = (
    output_path_data / "BEME_346_spatial_immunosenescence.h5ad"
)

output_file_355G = (
    output_path_data / "BEME_355G_spatial_immunosenescence.h5ad"
)

adata_346.write_h5ad(output_file_346)
adata_355G.write_h5ad(output_file_355G)

saved_346 = sc.read_h5ad(output_file_346, backed="r")
saved_355G = sc.read_h5ad(output_file_355G, backed="r")

print(saved_346)
print(saved_355G)

saved_346.file.close()
saved_355G.file.close()

AnnData object with n_obs × n_vars = 1388 × 22220 backed at '/content/drive/MyDrive/endo-immune-atlas/data/interim/spatial/BEME_346_spatial_immunosenescence.h5ad'
    obs: 'in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site', '_indices', '_scvi_batch', '_scvi_labels', 'meanscell_abundance_w_sf_B', 'meanscell_abundance_w_sf_CD4 T', 'meanscell_abundance_w_sf_CD8 T', 'meanscell_abundance_w_sf_DC', 'meanscell_abundance_w_sf_Mono-C', 'meanscell_abundance_w_sf_Mono-NC', 'meanscell_abundance_w_sf_NK-CD16+', 'meanscell_abundance_w_sf_NK-CD16-', 'meanscell_abundance_w_sf_TRM', 'meanscell_abundance_w_sf_Treg', 'meanscell_abundance_w_sf_γδ T', 'Total immune', 'B', 'CD4 T', 'CD8 T', 'DC', 'Mono-C', 'Mono-NC', 'NK-CD16+', 'NK-CD16-', 'TRM', 'Treg', 'γδ T', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'region_cluster', 'total_immune_spot_level', 'total_immune_region_level', 'tissue_level_immune_class', 'projected_senescence_